# 📓 Notebook 2 — Huấn luyện & Đánh giá

**Đề tài**: Phân loại rau sạch / rau hỏng bằng Deep Learning

Notebook này thực hiện:
1. Setup môi trường + load generators từ `dataset/{train,valid,test}`
2. **Huấn luyện MobileNetV2** (Transfer Learning, 2 pha: freeze → fine-tune)
3. **Huấn luyện ResNet50** (cùng chiến lược)
4. Đánh giá trên test set: accuracy / precision / recall / F1
5. **Trực quan hoá phong phú**: accuracy curve, loss curve, confusion matrix (raw + normalized), classification report heatmap, ROC curve, bar chart so sánh, ảnh sai phân loại
6. Demo predict trên ảnh thực tế

💡 Yêu cầu: đã chạy xong `01_prepare.ipynb`.

## 1. Setup

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path('..').resolve()
if Path.cwd().name == 'notebook':
    os.chdir(ROOT)
sys.path.insert(0, str(Path.cwd()))

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf
from tensorflow.keras import callbacks as cb, metrics as km
from tensorflow.keras.optimizers import Adam

# GPU memory growth (tránh chiếm hết VRAM laptop)
for gpu in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except: pass

tf.keras.utils.set_random_seed(42)
sns.set_theme(style='whitegrid')

print('TF:', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU') or '(CPU only)')
print('CWD:', Path.cwd())

In [ ]:
# Cấu hình chung
IMG_SIZE   = 224
BATCH      = 32      # giảm xuống 16 nếu OOM
EPOCHS_P1  = 15      # phase 1: freeze base, train head
EPOCHS_P2  = 8       # phase 2: fine-tune top layers

DATA_DIR   = Path('dataset')
CKPT_DIR   = Path('checkpoints'); CKPT_DIR.mkdir(exist_ok=True)
RESULTS    = Path('results');     RESULTS.mkdir(exist_ok=True)
LOGS       = Path('logs');        LOGS.mkdir(exist_ok=True)

from preprocessing.augmentation import build_train_generator, build_eval_generator
train_gen = build_train_generator(DATA_DIR/'train', img_size=IMG_SIZE, batch_size=BATCH)
valid_gen = build_eval_generator(DATA_DIR/'valid', img_size=IMG_SIZE, batch_size=BATCH)
test_gen  = build_eval_generator(DATA_DIR/'test',  img_size=IMG_SIZE, batch_size=BATCH, shuffle=False)
CLASS_NAMES = list(train_gen.class_indices.keys())
print(f'\n📦 Classes: {train_gen.class_indices}')
print(f'   train={train_gen.samples}, valid={valid_gen.samples}, test={test_gen.samples}')

## 2. Hàm tiện ích — train 2 pha

In [ ]:
def make_callbacks(name):
    return [
        cb.ModelCheckpoint(str(CKPT_DIR/f'{name}_best.keras'),
                           monitor='val_accuracy', mode='max',
                           save_best_only=True, verbose=1),
        cb.EarlyStopping(monitor='val_loss', patience=5,
                         restore_best_weights=True, verbose=1),
        cb.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                             patience=2, min_lr=1e-7, verbose=1),
        cb.CSVLogger(str(LOGS/f'{name}.csv'), append=False),
    ]

def compile_model(model, lr):
    model.compile(optimizer=Adam(lr),
                  loss='categorical_crossentropy',
                  metrics=['accuracy', km.Precision(name='precision'),
                                       km.Recall(name='recall')])

def merge_history(h1, h2):
    out = {k: list(v) for k, v in h1.history.items()}
    for k, v in h2.history.items():
        out.setdefault(k, []).extend(list(v))
    return out

def train_two_phase(builder, unfreezer, name, n_unfreeze):
    print(f'\n{"="*60}\n🏗️  Build {name}\n{"="*60}')
    model = builder(num_classes=2, img_size=IMG_SIZE)
    compile_model(model, lr=1e-3)
    print(f'   Total params: {model.count_params():,}')

    print(f'\n🔥 Phase 1: train classifier head (base frozen) — {EPOCHS_P1} epochs')
    h1 = model.fit(train_gen, validation_data=valid_gen,
                   epochs=EPOCHS_P1, callbacks=make_callbacks(name), verbose=2)

    print(f'\n🎯 Phase 2: fine-tune top {n_unfreeze} layers — {EPOCHS_P2} epochs')
    unfreezer(model, n_layers=n_unfreeze)
    compile_model(model, lr=1e-5)
    h2 = model.fit(train_gen, validation_data=valid_gen,
                   epochs=EPOCHS_P2, callbacks=make_callbacks(name), verbose=2)

    history = merge_history(h1, h2)
    with open(RESULTS/f'{name}_history.json', 'w') as f:
        json.dump(history, f, indent=2)
    return model, history

## 3. Huấn luyện MobileNetV2

In [ ]:
from models.mobilenet_model import build_mobilenet, unfreeze_for_finetune as unfreeze_mb
mb_model, mb_history = train_two_phase(build_mobilenet, unfreeze_mb,
                                       name='mobilenet', n_unfreeze=30)

## 4. Huấn luyện ResNet50

In [ ]:
from models.resnet_model import build_resnet, unfreeze_for_finetune as unfreeze_rn
rn_model, rn_history = train_two_phase(build_resnet, unfreeze_rn,
                                       name='resnet', n_unfreeze=40)

## 5. Đánh giá trên test set

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix,
                             roc_curve, auc)

def evaluate(model, name):
    test_gen.reset()
    y_prob = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = test_gen.classes
    metrics = {
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall':    recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1':        f1_score(y_true, y_pred, average='macro', zero_division=0),
    }
    with open(RESULTS/f'{name}_metrics.json', 'w') as f:
        json.dump(metrics, f, indent=2)
    return y_true, y_pred, y_prob, metrics

y_true_mb, y_pred_mb, y_prob_mb, m_mb = evaluate(mb_model, 'mobilenet')
y_true_rn, y_pred_rn, y_prob_rn, m_rn = evaluate(rn_model, 'resnet')

df_metrics = pd.DataFrame({'MobileNetV2': m_mb, 'ResNet50': m_rn}).T
print('\n📊 Metrics tổng hợp (test set):')
print(df_metrics.round(4).to_string())

## 6. Biểu đồ Accuracy & Loss curves

In [ ]:
def plot_history(hist, name, ax_acc, ax_loss):
    ep = range(1, len(hist['loss'])+1)
    ax_acc.plot(ep, hist['accuracy'], 'o-', label='train', ms=3)
    ax_acc.plot(ep, hist['val_accuracy'], 's-', label='valid', ms=3)
    ax_acc.set_title(f'{name} — Accuracy', fontweight='bold')
    ax_acc.set_xlabel('Epoch'); ax_acc.set_ylabel('Accuracy')
    ax_acc.legend(); ax_acc.grid(alpha=.3)
    ax_loss.plot(ep, hist['loss'], 'o-', label='train', ms=3)
    ax_loss.plot(ep, hist['val_loss'], 's-', label='valid', ms=3)
    ax_loss.set_title(f'{name} — Loss', fontweight='bold')
    ax_loss.set_xlabel('Epoch'); ax_loss.set_ylabel('Loss')
    ax_loss.legend(); ax_loss.grid(alpha=.3)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
plot_history(mb_history, 'MobileNetV2', axes[0,0], axes[0,1])
plot_history(rn_history, 'ResNet50',    axes[1,0], axes[1,1])
plt.tight_layout()
plt.savefig(RESULTS/'training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Confusion Matrix (raw + normalized)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for row, (name, y_t, y_p) in enumerate([
    ('MobileNetV2', y_true_mb, y_pred_mb),
    ('ResNet50',    y_true_rn, y_pred_rn),
]):
    cm = confusion_matrix(y_t, y_p)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[row,0])
    axes[row,0].set_title(f'{name} — CM (raw count)', fontweight='bold')
    axes[row,0].set_xlabel('Predicted'); axes[row,0].set_ylabel('True')
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens', cbar=False,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[row,1])
    axes[row,1].set_title(f'{name} — CM (normalized)', fontweight='bold')
    axes[row,1].set_xlabel('Predicted'); axes[row,1].set_ylabel('True')
plt.tight_layout()
plt.savefig(RESULTS/'confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Classification Report (heatmap)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (name, y_t, y_p) in zip(axes, [
    ('MobileNetV2', y_true_mb, y_pred_mb),
    ('ResNet50',    y_true_rn, y_pred_rn),
]):
    rep = classification_report(y_t, y_p, target_names=CLASS_NAMES,
                                output_dict=True, zero_division=0)
    df = pd.DataFrame(rep).iloc[:-1, :len(CLASS_NAMES)].T
    sns.heatmap(df, annot=True, fmt='.3f', cmap='YlGnBu', vmin=0.5, vmax=1.0,
                cbar=True, ax=ax)
    ax.set_title(f'{name} — Classification report', fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS/'classification_reports.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. ROC Curve + AUC

In [ ]:
plt.figure(figsize=(7, 6))
for name, y_t, y_pr, color in [
    ('MobileNetV2', y_true_mb, y_prob_mb, '#1976D2'),
    ('ResNet50',    y_true_rn, y_prob_rn, '#C62828'),
]:
    fpr, tpr, _ = roc_curve(y_t, y_pr[:, 1])
    a = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={a:.4f})')
plt.plot([0,1], [0,1], 'k--', alpha=.4, label='Random')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve — So sánh 2 mô hình', fontweight='bold')
plt.legend(loc='lower right'); plt.grid(alpha=.3)
plt.tight_layout()
plt.savefig(RESULTS/'roc_curve.png', dpi=120, bbox_inches='tight')
plt.show()

## 10. Bar chart so sánh 2 mô hình

In [ ]:
ax = df_metrics.plot.bar(figsize=(9, 4.5), rot=0, width=0.7,
                         colormap='viridis', edgecolor='white')
ax.set_title('So sánh MobileNetV2 vs ResNet50 trên Test set',
             fontweight='bold', fontsize=13)
ax.set_ylabel('Score'); ax.set_ylim(0, 1.05)
ax.legend(title='Metric', bbox_to_anchor=(1.02, 1), loc='upper left')
for c in ax.containers:
    ax.bar_label(c, fmt='%.3f', label_type='edge', fontsize=9)
plt.tight_layout()
plt.savefig(RESULTS/'comparison_bar.png', dpi=120, bbox_inches='tight')
plt.show()

## 11. Phân tích lỗi — ảnh bị phân loại sai

Hiển thị 8 ảnh bị MobileNetV2 + ResNet50 cùng dự đoán SAI (FN/FP). Dùng để hiểu pattern lỗi cho phần báo cáo.

In [ ]:
from PIL import Image
wrong_mb = np.where(y_true_mb != y_pred_mb)[0]
wrong_rn = np.where(y_true_rn != y_pred_rn)[0]
print(f'❌ MobileNet dự đoán sai: {len(wrong_mb)}/{len(y_true_mb)}')
print(f'❌ ResNet50  dự đoán sai: {len(wrong_rn)}/{len(y_true_rn)}')

filepaths = test_gen.filepaths
samples = wrong_mb[:8] if len(wrong_mb) >= 8 else wrong_mb
if len(samples) == 0:
    print('🎉 Không có ảnh nào sai (hoặc test set quá nhỏ)')
else:
    fig, axes = plt.subplots(2, 4, figsize=(13, 6.5))
    for ax, idx in zip(axes.ravel(), samples):
        img = Image.open(filepaths[idx])
        true_lbl = CLASS_NAMES[y_true_mb[idx]]
        pred_lbl = CLASS_NAMES[y_pred_mb[idx]]
        conf = y_prob_mb[idx, y_pred_mb[idx]]
        ax.imshow(img); ax.axis('off')
        ax.set_title(f'True: {true_lbl}\nPred: {pred_lbl} ({conf*100:.1f}%)',
                     color='red', fontsize=10)
    plt.suptitle('Ảnh MobileNetV2 phân loại SAI', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS/'misclassified.png', dpi=120, bbox_inches='tight')
    plt.show()

## 12. Demo predict trên ảnh thực tế

Chọn 6 ảnh ngẫu nhiên từ test set và xem 2 model dự đoán thế nào.

In [ ]:
import random
random.seed(7)
n_demo = 6
indices = random.sample(range(len(filepaths)), n_demo)

fig, axes = plt.subplots(2, n_demo, figsize=(3*n_demo, 6.5))
for col, idx in enumerate(indices):
    img = Image.open(filepaths[idx])
    true_lbl = CLASS_NAMES[y_true_mb[idx]]
    for row, (model_name, y_p, y_pr) in enumerate([
        ('MobileNetV2', y_pred_mb, y_prob_mb),
        ('ResNet50',    y_pred_rn, y_prob_rn),
    ]):
        pred_lbl = CLASS_NAMES[y_p[idx]]
        conf = y_pr[idx, y_p[idx]]
        ok = pred_lbl == true_lbl
        color = '#2E7D32' if ok else '#C62828'
        axes[row, col].imshow(img); axes[row, col].axis('off')
        axes[row, col].set_title(
            f'{model_name}\nTrue: {true_lbl}\nPred: {pred_lbl} ({conf*100:.1f}%)',
            color=color, fontsize=9, fontweight='bold' if ok else 'normal')
plt.suptitle('Demo predict trên 6 ảnh test', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS/'predict_demo.png', dpi=120, bbox_inches='tight')
plt.show()

## ✅ Tổng kết

Notebook đã sinh ra trong `results/`:

| File | Nội dung |
|---|---|
| `training_curves.png`        | Accuracy + Loss của 2 model |
| `confusion_matrices.png`     | CM raw + normalized cho 2 model |
| `classification_reports.png` | Precision/Recall/F1 per-class |
| `roc_curve.png`              | ROC + AUC so sánh |
| `comparison_bar.png`         | Bar chart 4 metrics |
| `misclassified.png`          | 8 ảnh bị sai để phân tích |
| `predict_demo.png`           | 6 ảnh demo predict |
| `mobilenet_history.json`     | Lịch sử train MobileNet |
| `resnet_history.json`        | Lịch sử train ResNet |
| `mobilenet_metrics.json`     | Metrics cuối cùng MobileNet |
| `resnet_metrics.json`        | Metrics cuối cùng ResNet |

Và checkpoint trong `checkpoints/`:
- `mobilenet_best.keras`
- `resnet_best.keras`

👉 **Dùng cho báo cáo**: chèn các hình từ `results/` vào các chương 4, 6 của `report/report_outline.md`.